# Pydantic Request / Response Validation

This notebook covers:

1. Define a request body with a Pydantic `BaseModel`
2. Define a typed response model and let FastAPI shape the response
3. Validate fields with `Field(...)` constraints and `@field_validator`
4. Return 422 automatically on bad input

**Scope**: FastAPI + Pydantic v2. Each section spins up its own tiny app + `TestClient` so examples stand alone.

We grow the `Asset` model across the notebook: minimal in §2, constrained in §4, validated in §5. By the end you have the schema the capstone uses verbatim.

## 1. Why Validate at the Boundary

Every API has a *boundary*: where untrusted bytes from the network become typed values your code reasons about. If garbage data gets past that boundary, every line downstream is debugging the wrong layer — your business logic ends up shadow-checking types it shouldn't have to.

Pydantic v2 moves parsing **and** validation to that boundary, declaratively. You describe the shape you want; Pydantic enforces it. FastAPI wires this in for you — the request body is parsed into the model before your handler runs, and on failure you get a structured 422 without writing a line of validation code.

Three properties make this the default for FastAPI apps:

- **Declarative** — the schema is the type hints, not a separate validator class.
- **Fast** — Pydantic v2's core is written in Rust; validation is a non-issue at API throughput.
- **Round-trippable** — the same model generates the OpenAPI schema, the docs, and (with `response_model`) shapes the outgoing JSON.

## 2. Request Body as a `BaseModel`

A `BaseModel` subclass declares the body shape. FastAPI infers from the type hint on the handler parameter that this argument is the request body (anything else with a primitive type is a query param).

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

class AssetIn(BaseModel):
    ticker: str
    name: str
    price: float

app = FastAPI()

@app.post("/assets")
def create_asset(asset: AssetIn):
    # Inside the handler, `asset` is a fully validated AssetIn instance.
    return {"received": asset.model_dump(), "ticker_upper": asset.ticker.upper()}

client = TestClient(app)

r = client.post("/assets", json={"ticker": "aapl", "name": "Apple Inc.", "price": 195.0})
print("status:", r.status_code)
print("body  :", r.json())

# Pydantic v2 still coerces obvious primitives — e.g., int -> float on `price`.
r = client.post("/assets", json={"ticker": "aapl", "name": "Apple Inc.", "price": 195})
print("\nint coerced to float:", r.json()["received"]["price"], type(r.json()["received"]["price"]).__name__)

## 3. `response_model` Shaping

You often want the *output* shape to differ from the internal record: hide internal IDs, strip credentials, omit timestamps the client doesn't need.

Set `response_model=` on the route. FastAPI runs the return value through that model, dropping anything not declared. This protects you from leaking new fields by accident — adding `internal_score` to the storage layer doesn't suddenly expose it to API clients.

In [ ]:
class AssetOut(BaseModel):
    ticker: str
    name: str
    price: float
    # Note: no `internal_score` here.

app = FastAPI()

@app.post("/assets", response_model=AssetOut)
def create_asset(asset: AssetIn):
    # Pretend we stored extra fields server-side.
    record = asset.model_dump() | {"internal_score": 0.91, "audit_id": "x-7421"}
    return record  # contains internal fields...

client = TestClient(app)

r = client.post("/assets", json={"ticker": "AAPL", "name": "Apple Inc.", "price": 195.0})
print("status        :", r.status_code)
print("body returned :", r.json())
print("note          : internal_score and audit_id were stripped by response_model")

## 4. Field Constraints (`min_length`, `gt`, `le`, …)

`Field(...)` adds constraints that Pydantic enforces and FastAPI advertises in the OpenAPI schema. Each constraint failure becomes a 422 with a precise pointer to the offending field.

The portfolio-domain rules we want for `Asset`:

- `ticker` — 1–10 uppercase letters (or `.`), no lowercase, no symbols.
- `name` — non-empty, capped at 100 chars.
- `price` — non-negative, ≤ a sane upper bound.

In [ ]:
from pydantic import Field

class AssetIn(BaseModel):
    ticker: str = Field(min_length=1, max_length=10, pattern=r"^[A-Z.]+$")
    name: str = Field(min_length=1, max_length=100)
    price: float = Field(ge=0, le=1_000_000)

app = FastAPI()

@app.post("/assets")
def create_asset(asset: AssetIn):
    return asset.model_dump()

client = TestClient(app)

cases = [
    ("valid",            {"ticker": "AAPL",  "name": "Apple Inc.", "price": 195.0}),
    ("lowercase ticker", {"ticker": "aapl",  "name": "Apple Inc.", "price": 195.0}),
    ("empty name",       {"ticker": "AAPL",  "name": "",            "price": 195.0}),
    ("negative price",   {"ticker": "AAPL",  "name": "Apple Inc.", "price": -1.0}),
    ("ticker too long",  {"ticker": "AAAAAAAAAAA", "name": "X",    "price": 1.0}),
]
for label, body in cases:
    r = client.post("/assets", json=body)
    print(f"{label:20} -> {r.status_code}", "" if r.status_code == 200 else r.json()["detail"][0]["msg"])

## 5. `@field_validator` and `@model_validator`

`Field()` constraints cover the common cases. For anything custom — normalization (upper-case a ticker), cross-field rules (price can't exceed cap by currency), conditional logic — use validator methods.

- **`@field_validator("name")`** runs on one field. `mode="before"` runs on the raw input; `mode="after"` (default) runs on the parsed value.
- **`@model_validator(mode="after")`** runs after all fields are parsed and can see the whole instance — perfect for cross-field rules.

In [ ]:
from pydantic import field_validator, model_validator

class AssetIn(BaseModel):
    ticker: str = Field(min_length=1, max_length=10)
    name: str = Field(min_length=1, max_length=100)
    price: float = Field(ge=0)

    @field_validator("ticker", mode="before")
    @classmethod
    def upper_ticker(cls, v: str) -> str:
        # Normalize before constraint checks run — so "aapl" passes after upper-casing.
        return v.upper() if isinstance(v, str) else v

    @model_validator(mode="after")
    def name_must_differ_from_ticker(self):
        if self.name.upper() == self.ticker:
            raise ValueError("name must differ from ticker")
        return self

app = FastAPI()

@app.post("/assets")
def create_asset(asset: AssetIn):
    return asset.model_dump()

client = TestClient(app)

cases = [
    ("lowercased ticker normalized", {"ticker": "aapl", "name": "Apple Inc.", "price": 195.0}),
    ("name == ticker rejected",      {"ticker": "AAPL", "name": "AAPL",       "price": 195.0}),
]
for label, body in cases:
    r = client.post("/assets", json=body)
    print(f"{label:32} -> {r.status_code}", r.json() if r.status_code == 200 else r.json()["detail"][0]["msg"])

## 6. The Automatic 422 Response

When validation fails, FastAPI returns `422 Unprocessable Entity` with a structured body. The shape is consistent across every validation error in your app:

```json
{
  "detail": [
    {"type": "...", "loc": ["body", "<field>"], "msg": "...", "input": "..."}
  ]
}
```

- `loc` is a path into the request — `["body", "price"]`, `["query", "limit"]`, `["path", "asset_id"]`.
- `type` is machine-readable (`missing`, `string_pattern_mismatch`, `greater_than_equal`, …).
- `msg` is human-readable.
- `input` is the bad value (handy for logging, dangerous to echo to untrusted users).

You can customize this shape with an exception handler — chapter 6.1 covers handlers in full. The preview below flattens errors into a simpler `{field: message}` map, which is friendlier for frontends to render.

In [ ]:
from fastapi import Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse

class AssetIn(BaseModel):
    ticker: str = Field(min_length=1, max_length=10, pattern=r"^[A-Z.]+$")
    name: str = Field(min_length=1)
    price: float = Field(ge=0)

app = FastAPI()

@app.exception_handler(RequestValidationError)
async def flat_validation_handler(request: Request, exc: RequestValidationError):
    # Flatten to {field: message} — friendlier for a UI to render inline.
    errors = {".".join(str(p) for p in e["loc"][1:]): e["msg"] for e in exc.errors()}
    return JSONResponse(status_code=422, content={"errors": errors})

@app.post("/assets")
def create_asset(asset: AssetIn):
    return asset.model_dump()

client = TestClient(app)

# Default shape (without the handler) — for reference, here's what the raw error tree looks like.
# We'll re-show it from the inside by inspecting one error.
r = client.post("/assets", json={"ticker": "aapl", "name": "", "price": -1})
print("custom flat shape:", r.json())

# What FastAPI's default would have looked like (peek at .errors() directly):
import pydantic
try:
    AssetIn(ticker="aapl", name="", price=-1)
except pydantic.ValidationError as e:
    print("\nraw pydantic error tree (one per failure):")
    for err in e.errors():
        print(" ", {k: err[k] for k in ("type", "loc", "msg")})

## Key Takeaways

- **Validate at the boundary.** Pydantic models on request bodies mean handler code can assume types are right.
- **Use `response_model`** to lock down what leaves the API. New internal fields don't accidentally become public.
- **`Field()` constraints** cover length, range, pattern. They appear in OpenAPI for free.
- **`@field_validator` / `@model_validator`** for normalization (upper-case a ticker) and cross-field rules. Run-order matters: `mode="before"` sees raw input, `mode="after"` sees parsed values.
- **422 is structured.** The default error shape has `type`, `loc`, `msg`, `input` per failure. Customize with an exception handler when your frontend needs a different shape (chapter 6.1).

This `AssetIn` / `AssetOut` pair is the one the capstone's `/assets` router uses — unchanged.

## Exercises

**1. Add a `Portfolio` model.** Validate:

- `name`: non-empty, ≤ 50 chars.
- `base_currency`: exactly one of `"USD"`, `"EUR"`, `"GBP"` (use `Literal` or an `Enum`).
- `holdings`: a list of `{"ticker": str, "shares": float}` where `shares > 0`.

Build a POST endpoint that accepts it and returns the count of holdings. Force a 422 by submitting `shares: 0`.

**2. Cross-field rule.** Add a `@model_validator(mode="after")` to `Portfolio` that rejects duplicate tickers in `holdings`. Confirm the error message points to the right place in `loc`.

**3. Strict mode.** By default Pydantic coerces `"195.0"` (string) to `195.0` (float) on a `price: float` field. Use `Field(strict=True)` on `price` to disable coercion, and confirm a string body now gets a 422 where it used to pass.